In [1]:
import matplotlib.pyplot as plt
import os
import pandas as pd
import scanpy as sc
import squidpy as sq
import cv2
import matplotlib.pyplot as plt
import lazyslide as zs
from ipywidgets import interact, FloatSlider
import numpy as np
import glob

from pathlib import Path

# import sys
# sys.path.append('/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/workflow/scripts')
# import coda
# import readwrite
# cfg = readwrite.config()

import sys
sys.path.append("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1")
from norkin_organoid.workflow.scripts.xenium.morphology_code.get_embeddings import (
    NorkinOrganoidDataset,
)

FEATURES_PATH = "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/organoids_h&e/features_sdata"

%matplotlib inline
%load_ext autoreload
%autoreload 2

/work/PRTNR/CHUV/DIR/rgottar1/spatial/conda_envs/lazyslide_env/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/conda_envs/lazyslide_env/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [2]:
# Load in the data
import glob
import os
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import spatialdata as sd

sdata_paths = glob.glob(os.path.join(FEATURES_PATH, "*.zarr"))  # Adjust path if needed, e.g., "path/to/*.h5ad"
sdatas_dict = {}
adatas_list = []

for file_path in tqdm(sdata_paths):
    organoid_id = os.path.basename(file_path).replace('_features.zarr', '')
    sdata = sd.read_zarr(file_path)
    sdata.tables['features_adata'].obs['organoid_id'] = organoid_id
    adatas_list.append(sdata.tables['features_adata'])
    
    sdatas_dict[organoid_id] = sdata

# Merge all AnnData objects
merged_adata = ad.concat(adatas_list, join='outer', index_unique=None)
# merged_sdata = sd.concatenate({sdata.tables['features_adata'].obs['organoid_id'] : sdata for sdata in  sdatas_list}, join="outer")

# print(f"Merged AnnData shape: {merged_adata.shape}")
# print(f"Organoid IDs: {merged_adata.obs['organoid_id'].unique()}")

100%|██████████| 491/491 [03:37<00:00,  2.25it/s]
/work/PRTNR/CHUV/DIR/rgottar1/spatial/conda_envs/lazyslide_env/lib/python3.11/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
def make_master_table(sdata):
    def delete_repeated_columns(df):
        return df.loc[:,~df.columns.duplicated()].copy()

    # make_master_table(sdata)
    tile_coords_xenium_absolute = sdata.shapes['tile_coords_xenium_absolute'] 
    tile_coords_xenium_absolute = tile_coords_xenium_absolute.rename(columns={'geometry': 'xenium_tile_geometry'})
    tile_coords_xenium_absolute = delete_repeated_columns(tile_coords_xenium_absolute)
    tile_coords_hne_absolute = sdata.shapes['tile_coords_hne_absolute'] 
    tile_coords_hne_absolute = tile_coords_hne_absolute.rename(columns={'geometry': 'hne_tile_geometry'})
    metadata_table = sdata['features_adata'].obs

    # metadata_table = metadata_table.merge(tile_coords_xenium_absolute, on=["tile_id", "tissue_id"], how="inner")
    metadata_table = metadata_table.merge(tile_coords_hne_absolute, on=["tile_id", "tissue_id"], how="inner")
    metadata_table = metadata_table.merge(tile_coords_xenium_absolute, on=["tile_id", "tissue_id"], how="inner")
    metadata_table['morphological_features'] = list(sdata.tables['features_adata'].X)

    cells_in_tile = sdata.shapes['cells_in_tile']
    cells_in_tile = cells_in_tile.rename(columns={"geometry": "cell_geometry"})
    cells_in_tile = cells_in_tile.drop(["index_right", 'component_and_cluster_and_lasso'], axis=1)

    return cells_in_tile.merge(metadata_table, on=["tile_id", "tissue_id"], how="inner")


In [ ]:
master_dfs = []

for sdata in tqdm(sdatas_dict.values()):
    try: 
        df = make_master_table(sdata)
        master_dfs.append(df)
    except Exception: 
        pass


 23%|██▎       | 111/491 [00:00<00:02, 181.95it/s]

> /tmp/ipykernel_402492/3118647228.py(25)make_master_table()
     21         cells_in_tile = cells_in_tile.drop(["index_right", 'component_and_cluster_and_lasso'], axis=1)
     22     except Exception as e:
     23         import pdb; pdb.set_trace()
     24 
---> 25     return cells_in_tile.merge(metadata_table, on=["tile_id", "tissue_id"], how="inner")



100%|██████████| 491/491 [00:04<00:00, 101.91it/s]


[     cell_id                                      cell_geometry  \
 0        182  MULTIPOLYGON (((2167 2396, 2167 2395, 2168 239...   
 1        182  MULTIPOLYGON (((2167 2396, 2167 2395, 2168 239...   
 2        182  MULTIPOLYGON (((2167 2396, 2167 2395, 2168 239...   
 3        205  MULTIPOLYGON (((2064 2360, 2064 2362, 2066 236...   
 4        215  MULTIPOLYGON (((2165 2307, 2165 2308, 2164 230...   
 ..       ...                                                ...   
 572    13851  MULTIPOLYGON (((2110 2402, 2110 2403, 2111 240...   
 573    13851  MULTIPOLYGON (((2110 2402, 2110 2403, 2111 240...   
 574    13857  MULTIPOLYGON (((2112 2328, 2112 2332, 2115 233...   
 575    13857  MULTIPOLYGON (((2112 2328, 2112 2332, 2115 233...   
 576    13857  MULTIPOLYGON (((2112 2328, 2112 2332, 2115 233...   
 
                                           full_cell_id patient_id  \
 0    proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...       1HVQ   
 1    proseg_expected_CRC_PDO_hImmune_v1_d

In [95]:
master_df = pd.concat(master_dfs, ignore_index=True)
master_df.to_parquet("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/organoids_h&e/all_data.parquet", index=False)

In [98]:
df = pd.read_parquet("/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/organoids_h&e/all_data.parquet")
print(df.columns)
df.head()

Index(['cell_id', 'cell_geometry', 'full_cell_id', 'patient_id', 'method',
       'joint_id', 'tile_id', 'tissue_id', 'organoid_id', 'hne_tile_geometry',
       'xenium_tile_geometry', 'morphological_features'],
      dtype='object')


,cell_id,cell_geometry,full_cell_id,patient_id,method,joint_id,tile_id,tissue_id,organoid_id,hne_tile_geometry,xenium_tile_geometry,morphological_features
0,182,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,2,0,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...","[0.2668, 0.152, 0.1252, 0.02736, 0.05884, -0.2..."
1,182,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,1,0,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"[0.03035, 0.0901, 0.0898, -0.02246, 0.1653, -0..."
2,182,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,0,0,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"[0.2262, 0.0764, 0.2095, 0.0673, 0.061, -0.279..."
3,205,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,1,0,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"[0.03035, 0.0901, 0.0898, -0.02246, 0.1653, -0..."
4,215,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,2,0,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...","[0.2668, 0.152, 0.1252, 0.02736, 0.05884, -0.2..."


In [ ]:
cells_in_tile = sdata.shapes['cells_in_tile']
cells_in_tile = cells_in_tile.rename(columns={"geometry": "cell_geometry"})
cells_in_tile = cells_in_tile.drop(["index_right", 'component_and_cluster_and_lasso'], axis=1)
cells_in_tile.head()

,cell_id,cell_geometry,full_cell_id,patient_id,method,joint_id,tile_id,tissue_id
8,8,"MULTIPOLYGON (((2080 1959, 2080 1958, 2081 195...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1J25_o...,1J25,hImmune_v1_dapi,hImmune_v1_dapi__1J25,1,2
8,8,"MULTIPOLYGON (((2080 1959, 2080 1958, 2081 195...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1J25_o...,1J25,hImmune_v1_dapi,hImmune_v1_dapi__1J25,0,2
159,159,"MULTIPOLYGON (((1995 1889, 1995 1888, 1997 188...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1J25_o...,1J25,hImmune_v1_dapi,hImmune_v1_dapi__1J25,3,2
159,159,"MULTIPOLYGON (((1995 1889, 1995 1888, 1997 188...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1J25_o...,1J25,hImmune_v1_dapi,hImmune_v1_dapi__1J25,2,2
159,159,"MULTIPOLYGON (((1995 1889, 1995 1888, 1997 188...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1J25_o...,1J25,hImmune_v1_dapi,hImmune_v1_dapi__1J25,1,2


In [ ]:
sample_organoid_id, sample_sdata = list(sdatas_dict.items())[0]
sample_sdata['cells_in_tile']
#full_cell_id, tile_id, tissue_id, component_and_cluster_and_lasso

,cell_id,geometry,full_cell_id,patient_id,method,joint_id,component_and_cluster_and_lasso,index_right,tile_id,tissue_id
182,182,"MULTIPOLYGON (((2167 2396, 2167 2395, 2168 239...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,2,2,0
182,182,"MULTIPOLYGON (((2167 2396, 2167 2395, 2168 239...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1,1,0
182,182,"MULTIPOLYGON (((2167 2396, 2167 2395, 2168 239...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,0,0,0
205,205,"MULTIPOLYGON (((2064 2360, 2064 2362, 2066 236...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1,1,0
215,215,"MULTIPOLYGON (((2165 2307, 2165 2308, 2164 230...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,2,2,0
...,...,...,...,...,...,...,...,...,...,...
13851,13851,"MULTIPOLYGON (((2110 2402, 2110 2403, 2111 240...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1,1,0
13851,13851,"MULTIPOLYGON (((2110 2402, 2110 2403, 2111 240...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,0,0,0
13857,13857,"MULTIPOLYGON (((2112 2328, 2112 2332, 2115 233...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,2,2,0
13857,13857,"MULTIPOLYGON (((2112 2328, 2112 2332, 2115 233...",proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1HVQ,hImmune_v1_dapi,hImmune_v1_dapi__1HVQ,proseg_expected_CRC_PDO_hImmune_v1_dapi_1HVQ_o...,1,1,0


In [ ]:
# output format: cell, tile, organoid, representation
entries = {}

for organoid_id, sdata in sdatas_dict.items():
    for entry in sdata.tables['cell']